# 03 — QC Metrics & Normalization

Calculates Z', %CV, Signal Window and normalizes experimental data to controls.

**Requires:** `data/processed/all_plates_raw.csv`

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np

df = pd.read_csv('data/processed/all_plates_raw.csv')
plates = df['Plate'].unique()
print(f'Plates: {list(plates)}')
print(f'Total wells: {len(df)}')

Plates: ['Plate 1', 'Plate 2', 'Plate 3', 'Plate 4', 'Plate 5']
Total wells: 1920


## QC Metric Formulas

| Metric | Formula | Pass Threshold |
|---|---|---|
| Z' | `1 - (3σ_high + 3σ_low) / abs(μ_high - μ_low)` | > 0.5 |
| %CV Low | `(σ_low / μ_low) × 100` | < 10% |
| %CV High | `(σ_high / μ_high) × 100` | < 10% |
| Signal Window | `(μ_high - 3σ_high - μ_low - 3σ_low) / σ_low` | > 2 |

In [2]:
def calc_qc(low_rfu, high_rfu):
    mu_l, sd_l = low_rfu.mean(), low_rfu.std()
    mu_h, sd_h = high_rfu.mean(), high_rfu.std()
    zprime = 1 - (3*sd_h + 3*sd_l) / abs(mu_h - mu_l)
    cv_low  = sd_l / mu_l * 100
    cv_high = sd_h / mu_h * 100
    sw = ((mu_h - 3*sd_h) - (mu_l + 3*sd_l)) / sd_l
    return {
        'Mean_Low_RFU':  round(mu_l, 1),
        'SD_Low_RFU':    round(sd_l, 1),
        'Mean_High_RFU': round(mu_h, 1),
        'SD_High_RFU':   round(sd_h, 1),
        "Z_prime":       round(zprime, 3),
        'CV_Low_pct':    round(cv_low, 2),
        'CV_High_pct':   round(cv_high, 2),
        'Signal_Window': round(sw, 2),
        'Z_prime_Pass':  'PASS' if zprime > 0.5 else 'FAIL',
        'CV_Low_Pass':   'PASS' if cv_low < 10 else 'FAIL',
        'CV_High_Pass':  'PASS' if cv_high < 10 else 'FAIL',
        'SW_Pass':       'PASS' if sw > 2 else 'FAIL'
    }

## Calculate QC & normalize per plate

In [3]:
qc_rows = []
norm_records = []

for plate in plates:
    pdf = df[df['Plate'] == plate]
    low_rfu  = pdf[pdf['WellType'] == 'Low']['RFU'].values.astype(float)
    high_rfu = pdf[pdf['WellType'] == 'High']['RFU'].values.astype(float)

    qc = calc_qc(low_rfu, high_rfu)
    qc['Plate'] = plate
    qc_rows.append(qc)

    mu_low  = low_rfu.mean()
    mu_high = high_rfu.mean()

    for _, row in pdf.iterrows():
        pct = round((row['RFU'] - mu_low) / (mu_high - mu_low) * 100, 2)
        norm_records.append({
            'Plate': plate, 'Row': row['Row'], 'Column': row['Column'],
            'Well': row['Well'], 'WellType': row['WellType'],
            'RFU': row['RFU'], 'Pct_Activity': pct
        })

qc_df   = pd.DataFrame(qc_rows).set_index('Plate')
norm_df = pd.DataFrame(norm_records)

print(qc_df[['Z_prime', 'CV_Low_pct', 'CV_High_pct', 'Signal_Window',
             'Z_prime_Pass', 'CV_Low_Pass', 'CV_High_Pass', 'SW_Pass']].to_string())

         Z_prime  CV_Low_pct  CV_High_pct  Signal_Window Z_prime_Pass CV_Low_Pass CV_High_Pass SW_Pass
Plate                                                                                                 
Plate 1    0.795       17.07         6.60         465.08         PASS        FAIL         PASS    PASS
Plate 2    0.768       24.79         7.41         292.64         PASS        FAIL         PASS    PASS
Plate 3    0.792       23.83         6.63         333.85         PASS        FAIL         PASS    PASS
Plate 4    0.820       26.09         5.70         322.64         PASS        FAIL         PASS    PASS
Plate 5    0.776       28.04         7.12         279.40         PASS        FAIL         PASS    PASS


## Save outputs

In [4]:
qc_df.to_csv('data/processed/qc_metrics.csv')
norm_df.to_csv('data/processed/all_plates_normalized.csv', index=False)
print('Saved: data/processed/qc_metrics.csv')
print('Saved: data/processed/all_plates_normalized.csv')
norm_df.head()

Saved: data/processed/qc_metrics.csv
Saved: data/processed/all_plates_normalized.csv


,Plate,Row,Column,Well,WellType,RFU,Pct_Activity
0,Plate 1,A,1,A01,Low,339,0.13
1,Plate 1,A,2,A02,Medium,9723,31.49
2,Plate 1,A,3,A03,Experimental,20181,66.43
3,Plate 1,A,4,A04,Experimental,27184,89.83
4,Plate 1,A,5,A05,Experimental,13126,42.86
